In [ ]:
import torch
import torchvision
from torch import nn
from lightly.loss import NTXentLoss
from lightly.models.modules import SimCLRProjectionHead
from lightly.transforms.simclr_transform import SimCLRTransform
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import datasets, transforms
from torchvision.transforms import v2
from torch.utils.data import DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR
import math

torch.backends.cudnn.benchmark = True

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)
    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x += residual
        return F.relu(x)

class Encoder(nn.Module):
    def __init__(self, input_channels, flattened_size):
        super().__init__()
        self.backbone = torch.nn.Sequential( 
            torch.nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            torch.nn.BatchNorm2d(32),
            torch.nn.ReLU(),
            # Extra convolutional block added
            torch.nn.Conv2d(32, 32, kernel_size=3, padding=1),
            torch.nn.BatchNorm2d(32),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2, 2),
            torch.nn.Conv2d(32, 64, kernel_size=3, padding=1),
            torch.nn.BatchNorm2d(64),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2, 2),
            torch.nn.Dropout2d(0.5),
            torch.nn.Flatten(), 
            torch.nn.Linear(flattened_size, 2048)
        )
        self.projection_head = SimCLRProjectionHead(2048, 2048, 128)
        # self.projection_head = nn.Sequential(
        #     nn.Linear(2048, 2048),
        #     nn.ReLU(),
        #     SimCLRProjectionHead(2048, 2048, 128)
        # )
        # Initialize weights
        self.apply(self._init_weights)

    def forward(self, x):
        x = self.backbone(x).flatten(start_dim=1)
        z = self.projection_head(x)
        return z
    

# class Encoder(nn.Module):
#     def __init__(self, input_channels=3):
#         super().__init__()
#         self.backbone = nn.Sequential(
#             nn.Conv2d(input_channels, 32, 3, padding=1),  # Layer 1
#             nn.BatchNorm2d(32),
#             nn.ReLU(),
#             ResidualBlock(32),                    # Layers 2-3
#             nn.MaxPool2d(2),
#             nn.Conv2d(32, 64, 3, padding=1),     # Layer 4
#             nn.BatchNorm2d(64),
#             nn.ReLU(),
#             ResidualBlock(64),                   # Layers 5-6
#             nn.MaxPool2d(2),
#             nn.Dropout2d(0.3),
#             nn.Flatten(),
#             nn.Linear(64 * 8 * 8, 2048)          # Layer 7
#         )
#         self.projection_head = SimCLRProjectionHead(2048, 512, 128)
#         self.apply(self._init_weights)

#     def forward(self, x):
#         x = self.backbone(x)
#         z = self.projection_head(x)
#         return z

    
    def _init_weights(self, m):
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
    # def _init_weights(self, m):
    #     if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
    #         nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    #     if hasattr(m, 'bias') and m.bias is not None:
    #         nn.init.constant_(m.bias, 0)


class Classifier(nn.Module):
    def __init__(self, encoder, input_channels, flattened_size):
        super(Classifier, self).__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(128, 2048),
            nn.BatchNorm1d(2048),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(2048, 10)
        )
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.BatchNorm1d):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.encoder(x)
        x = self.classifier(x)
        return x


# class SimCLR(nn.Module):
#     def __init__(self):
#         super().__init__()
#         resnet = torchvision.models.resnet18()
#         self.backbone = nn.Sequential(*list(resnet.children())[:-1])
#         self.projection_head = SimCLRProjectionHead(512, 512, 128)

#     def forward(self, x):
#         x = self.backbone(x).flatten(start_dim=1)
#         z = self.projection_head(x)
#         return z

# Set up the model
input_channels = 3
flattened_size = 64 * 8 * 8  # Assuming input size is 32x32 and two max-pooling layers with stride 2
model = Encoder(input_channels, flattened_size)
# model = Encoder(input_channels=input_channels)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Set up the data
# simclr_transform = SimCLRTransform(
#     input_size=32,
#     gaussian_blur=0.5,
#     cj_strength=0.5,  # Increased from 0.5
#     random_gray_scale=0.2,
#     hf_prob=0.5,  # Add horizontal flip
#     vf_prob=0.5   # Add vertical flip
# )
simclr_transform = SimCLRTransform(
    input_size=32,
    cj_strength=0.8,
    gaussian_blur=0.7,
    random_gray_scale=0.3,
    hf_prob=0.5,
    vf_prob=0.5,
)
train_dataset_simclr = torchvision.datasets.CIFAR10(
    "datasets/cifar10", train=True, download=False, transform=simclr_transform
)
# Set up the loss and optimizer
criterion = NTXentLoss(temperature=0.1)

from torch.utils.data import Dataset
class MemoryDataset(Dataset):
    def __init__(self, dataset):
        self.data = [dataset[i] for i in range(len(dataset))]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]
train_dataset_simclr = MemoryDataset(train_dataset_simclr) # loads dataset to memory!
print(f"Preloaded {len(train_dataset_simclr)} samples into memory")
batch_size = 1024  # increased batch size
train_loader_simclr = torch.utils.data.DataLoader(
    train_dataset_simclr, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=0, pin_memory=True
)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)  # lower initial lr

print("starting training")
# Training loop
num_epochs = 150
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=0.01,
    steps_per_epoch=len(train_loader_simclr),
    epochs=num_epochs,
    pct_start=0.3  # 30% of training for warm-up
)
print("starting epochs")
for epoch in range(num_epochs):
    total_loss = 0
    for batch_idx, batch in enumerate(train_loader_simclr):
        # print(f"Processing batch {batch_idx}")
        x0, x1 = batch[0]
        # x0, x1 = x0.to(device), x1.to(device)
        x0, x1 = x0.to(device, non_blocking=True), x1.to(device, non_blocking=True)
        z0, z1 = model(x0), model(x1)
        loss = criterion(z0, z1)
        total_loss += loss.detach()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    avg_loss = total_loss / len(train_loader_simclr)
    print(f"Epoch: {epoch+1}/{num_epochs}, Loss: {avg_loss:.5f}")
    

    # Save model checkpoint at the end of each epoch
    if epoch == 0 or avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss,
        }, 'best_simclr_model.pth')
        print(f"Model saved at epoch {epoch+1} with loss: {avg_loss:.5f}")


c:\Users\gal19\anaconda3\envs\cs236781-hw\lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Preloaded 50000 samples into memory
starting training
starting epochs
Epoch: 1/150, Loss: 7.06868
Model saved at epoch 1 with loss: 7.06868
Epoch: 2/150, Loss: 6.64211
Model saved at epoch 2 with loss: 6.64211
Epoch: 3/150, Loss: 6.38326
Model saved at epoch 3 with loss: 6.38326
Epoch: 4/150, Loss: 6.19825
Model saved at epoch 4 with loss: 6.19825
Epoch: 5/150, Loss: 6.02473
Model saved at epoch 5 with loss: 6.02473
Epoch: 6/150, Loss: 5.89120
Model saved at epoch 6 with loss: 5.89120
Epoch: 7/150, Loss: 5.75375
Model saved at epoch 7 with loss: 5.75375
Epoch: 8/150, Loss: 5.64444
Model saved at epoch 8 with loss: 5.64444
Epoch: 9/150, Loss: 5.57545
Model saved at epoch 9 with loss: 5.57545
Epoch: 10/150, Loss: 5.46703
Model saved at epoch 10 with loss: 5.46703
Epoch: 11/150, Loss: 5.35282
Model saved at epoch 11 with loss: 5.35282
Epoch: 12/150, Loss: 5.26463
Model saved at epoch 12 with loss: 5.26463
Epoch: 13/150, Loss: 5.18658
Model saved at epoch 13 with loss: 5.18658
Epoch: 14/15

In [ ]:
# batch_size = 1024  # increased batch size
# train_loader_simclr = torch.utils.data.DataLoader(
#     train_dataset_simclr, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=0, pin_memory=True
# )
# optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)  # lower initial lr
# print("starting training")
# # Training loop
# num_epochs = 100
# scheduler = torch.optim.lr_scheduler.OneCycleLR(
#     optimizer, 
#     max_lr=0.01,
#     steps_per_epoch=len(train_loader_simclr),
#     epochs=num_epochs,
#     pct_start=0.3  # 30% of training for warm-up
# )
# print("starting epochs")
# for epoch in range(num_epochs):
#     total_loss = 0
#     for batch_idx, batch in enumerate(train_loader_simclr):
#         # print(f"Processing batch {batch_idx}")
#         x0, x1 = batch[0]
#         x0, x1 = x0.to(device), x1.to(device)
#         # x0, x1 = x0.to(device, non_blocking=True), x1.to(device, non_blocking=True)
#         z0, z1 = model(x0), model(x1)
#         loss = criterion(z0, z1)
#         total_loss += loss.detach()
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

#         optimizer.step()
#         scheduler.step()
#         optimizer.zero_grad()
#     avg_loss = total_loss / len(train_loader_simclr)
#     print(f"Epoch: {epoch+1}/{num_epochs}, Loss: {avg_loss:.5f}")
    

#     # Save model checkpoint at the end of each epoch
#     if epoch == 0 or avg_loss < best_loss:
#         best_loss = avg_loss
#         torch.save({
#             'epoch': epoch,
#             'model_state_dict': model.state_dict(),
#             'optimizer_state_dict': optimizer.state_dict(),
#             'loss': avg_loss,
#         }, 'best_simclr_model.pth')
#         print(f"Model saved at epoch {epoch+1} with loss: {avg_loss:.5f}")


In [32]:
mean = [0.4914, 0.4822, 0.4465]
std = [0.2023, 0.1994, 0.2010]
config = {
    'MNIST': {
        'input_channels': 1,
        'input_size': 28,
        'num_classes': 10,
        'train_transform': transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ]),
        'val_transform': transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ]),
        'mean': (0.1307,),
        'std': (0.3081,)
    },
    'CIFAR10': {
        'input_channels': 3,
        'input_size': 32,
        'num_classes': 10,
        'train_transform': transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std)
        ]),
        'val_transform': transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std)
        ]),
        'mean': (0.4914, 0.4822, 0.4465),
        'std': (0.2023, 0.1994, 0.2010)
    }
}


In [35]:
def load_dataset(dataset_name):
    cfg = config[dataset_name]
    
    if dataset_name == 'MNIST':
        train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=cfg['train_transform'])
        test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=cfg['val_transform'])
    elif dataset_name == 'CIFAR10':
        train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=cfg['train_transform'])
        test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=cfg['val_transform'])
    
    # Split train into train and validation
    train_size = len(train_dataset) - 10000
    train_dataset, val_dataset = random_split(train_dataset, [train_size, 10000])
    
    # Apply validation transform to the validation dataset
    val_dataset.dataset.transform = cfg['val_transform']
    
    return train_dataset, val_dataset, test_dataset

def create_data_loaders(train_dataset, val_dataset, test_dataset, batch_size):
    return (
        DataLoader(train_dataset, batch_size=batch_size, shuffle=True),
        DataLoader(val_dataset, batch_size=batch_size, shuffle=False),
        DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    )

In [36]:
# Load the CIFAR-10 test set with standard transforms for evaluation
test_transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2470, 0.2435, 0.2616]
    )
])

# Data preparation (example for CIFAR-10)
mean = [0.4914, 0.4822, 0.4465]
std = [0.2023, 0.1994, 0.2010]
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

batch_size=1024
dataset_name = "CIFAR10"
cfg = config[dataset_name]
train_dataset, val_dataset, test_dataset = load_dataset(dataset_name)
train_loader, val_loader, test_loader = create_data_loaders(train_dataset, val_dataset, test_dataset, batch_size)


In [41]:
checkpoint = torch.load('best_simclr_model.pth')
# model = Encoder(input_channels=input_channels, flattened_size=(64 * 8 * 8))
# device = "cuda" if torch.cuda.is_available() else "cpu"
# model.to(device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded encoder from epoch {checkpoint['epoch']} with loss: {checkpoint['loss']:.5f}")

Loaded encoder from epoch 149 with loss: 1.68463


In [ ]:
import importlib
import utils
importlib.reload(utils)
from utils import plot_tsne
model.eval()  # Set to evaluation mode
plot_tsne(model.backbone, test_loader, device) #should send the backbone or projection head?
# plot_tsne(model, test_loader, device)

In [44]:
def train_classifier(model, train_loader, val_loader, device, num_epochs=15):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=5e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=5)
    best_acc = 0.0
    patience_counter = 0
    patience_limit = 10

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        # Validation phase
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        val_acc = 100 * correct / total
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}, Val Accuracy: {val_acc:.2f}%")
        
        # Scheduler step
        scheduler.step(val_acc)
        
        # Early stopping
        if val_acc > best_acc:
            best_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience_limit:
                print("Early stopping triggered.")
                break
    
    return model

def evaluate_classifier(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return 100. * correct / total

In [46]:
# Load pretrained SimCLR encoder
checkpoint = torch.load('best_simclr_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded encoder from epoch {checkpoint['epoch']} with loss: {checkpoint['loss']:.5f}")

# Initialize Classifier with Pretrained Encoder
classifier = Classifier(model, input_channels, flattened_size).to(device)

# Freeze Encoder (optional, comment out to fine-tune the entire model)
for param in classifier.encoder.parameters():
    param.requires_grad = False

# Train the model
classifier = train_classifier(classifier, train_loader, val_loader, device, num_epochs=25)



Loaded encoder from epoch 149 with loss: 1.68463
Epoch 1/25, Loss: 21.2413, Val Accuracy: 10.09%
Epoch 2/25, Loss: 18.5402, Val Accuracy: 10.09%


KeyboardInterrupt: 

In [11]:
classifier.load_state_dict(torch.load('best_model.pth'))
classifier.eval()

correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = classifier(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
test_acc = 100 * correct / total
print(f"Test Accuracy: {test_acc:.2f}%")

Test Accuracy: 46.01%
